In [2]:
import json, re
from pathlib import Path

txt = Path("viz/series_data.js").read_text()
data = json.loads(re.sub(r"^window\.SERIES_DATA = |;\s*$", "", txt.strip()))
thr = data["models"][0]["threshold"]
print(f"threshold: {thr} | #series: {len(data['series'])}\n")
for s in data["series"]:
    scores = [w["scores"]["if"] for w in s["weeks"] if w["scores"]["if"] is not None]
    flags = [x for x in scores if x >= thr]
    mx = max(scores) if scores else float("nan")
    print(f"{s['id'][:30]:32s} weeks={len(s['weeks']):3d} scored={len(scores):3d} "
          f"max={mx:.3f} flags={len(flags)}")

threshold: 0.5505 | #series: 12

FOODS_3_586_TX_1_evaluation      weeks=277 scored=225 max=0.507 flags=0
FOODS_3_097_CA_1_evaluation      weeks=277 scored=211 max=0.612 flags=6
FOODS_2_386_CA_4_evaluation      weeks=224 scored=130 max=0.695 flags=10
FOODS_3_587_CA_3_evaluation      weeks=277 scored=213 max=0.727 flags=48
HOBBIES_1_323_TX_2_evaluation    weeks=277 scored=221 max=0.579 flags=7
HOBBIES_1_005_TX_1_evaluation    weeks=263 scored=160 max=0.640 flags=30
HOBBIES_1_278_CA_1_evaluation    weeks=277 scored=147 max=0.714 flags=41
HOBBIES_2_059_CA_2_evaluation    weeks=277 scored=182 max=0.711 flags=74
HOUSEHOLD_1_225_TX_1_evaluatio   weeks=195 scored=143 max=0.548 flags=0
HOUSEHOLD_1_334_WI_2_evaluatio   weeks=208 scored=156 max=0.641 flags=7
HOUSEHOLD_2_366_TX_1_evaluatio   weeks=277 scored=217 max=0.669 flags=11
HOUSEHOLD_1_437_CA_3_evaluatio   weeks=277 scored=206 max=0.715 flags=56


In [3]:
from src.models.autoencoder import build_training_windows, build_series_channels, WINDOW_L

X = build_training_windows(sample_per_series=50)
print("windows:", X.shape)                 # (~700k, 13, 3)
print("sales_norm  mean/std:", X[:, :, 0].mean().round(3), X[:, :, 0].std().round(3))
print("sin range:", X[:, :, 1].min().round(2), X[:, :, 1].max().round(2))
print("cos range:", X[:, :, 2].min().round(2), X[:, :, 2].max().round(2))

windows: (774044, 13, 3)
sales_norm  mean/std: 0.082 0.838
sin range: -1.0 1.0
cos range: -1.0 1.0


In [1]:
import torch
from src.models.autoencoder import LSTMAutoencoder, get_device

dev = get_device()
print("device:", dev)

model = LSTMAutoencoder().to(dev)
xb = torch.randn(8, 13, 3, device=dev)
out = model(xb)
print("out shape:", out.shape)                 # torch.Size([8, 13, 3])
print("params:", sum(p.numel() for p in model.parameters()))

device: mps
out shape: torch.Size([8, 13, 3])
params: 14355


In [1]:
import time
from src.models.autoencoder import build_training_windows, train_autoencoder

X = build_training_windows(sample_per_series=50)   # ~770k fenêtres
print("X:", X.shape)

t0 = time.time()
model, best = train_autoencoder(X, epochs=3)        # court : juste pour jauger
print(f"3 epochs in {time.time()-t0:.0f}s  | best val {best:.4f}")

X: (774044, 13, 3)
epoch  1 train 0.1496 val 0.1103
epoch  2 train 0.1037 val 0.0966
epoch  3 train 0.0969 val 0.0923
3 epochs in 46s  | best val 0.0923


In [5]:
from src.models.autoencoder import train_autoencoder, save_ae, build_training_windows

X = build_training_windows(sample_per_series=50)   # ~770k fenêtres
print("X:", X.shape)

model, best = train_autoencoder(X, epochs=80, patience=8, latent=4)   # early stopping s'arrêtera avant si besoin
print("best val:", round(best, 4))

out = save_ae(model, best)
print("saved to", out)

X: (774044, 13, 3)
epoch  1 train 0.1526 val 0.1114
epoch  2 train 0.1116 val 0.1088
epoch  3 train 0.1099 val 0.1081
epoch  4 train 0.1093 val 0.1082
epoch  5 train 0.1087 val 0.1073
epoch  6 train 0.1083 val 0.1073
epoch  7 train 0.1080 val 0.1064
epoch  8 train 0.1075 val 0.1071
epoch  9 train 0.1074 val 0.1061
epoch 10 train 0.1070 val 0.1053
epoch 11 train 0.1067 val 0.1051
epoch 12 train 0.1063 val 0.1046
epoch 13 train 0.1058 val 0.1054
epoch 14 train 0.1054 val 0.1052
epoch 15 train 0.1050 val 0.1036
epoch 16 train 0.1046 val 0.1029
epoch 17 train 0.1042 val 0.1023
epoch 18 train 0.1035 val 0.1059
epoch 19 train 0.1030 val 0.1020
epoch 20 train 0.1029 val 0.1015
epoch 21 train 0.1014 val 0.1000
epoch 22 train 0.1004 val 0.0980
epoch 23 train 0.0992 val 0.0975
epoch 24 train 0.0974 val 0.0959
epoch 25 train 0.0956 val 0.0959
epoch 26 train 0.0950 val 0.0926
epoch 27 train 0.0941 val 0.0962
epoch 28 train 0.0937 val 0.0914
epoch 29 train 0.0927 val 0.0909
epoch 30 train 0.0922 va

In [6]:
import numpy as np
import pandas as pd
from src.config import DATA_DIR
from src.models.autoencoder import (
    load_ae, reconstruction_error, build_series_channels, make_windows, get_device
)

dev = get_device()
model, cfg = load_ae()
print("loaded AE, latent =", cfg["latent"])

# fenêtres propres depuis un échantillon de séries
feats = pd.read_parquet(DATA_DIR / "features.parquet")
sample_ids = feats["id"].drop_duplicates().sample(300, random_state=0)
clean = np.concatenate([
    make_windows(build_series_channels(s)[0])[0]
    for _, s in feats[feats.id.isin(sample_ids)].sort_values("wm_yr_wk").groupby("id")
]).astype(np.float32)

# copies anormales : un pic injecté au centre de chaque fenêtre (canal ventes)
anom = clean.copy()
L = clean.shape[1]
anom[:, L // 2, 0] += 5.0

err_clean = reconstruction_error(model, clean, dev)
err_anom = reconstruction_error(model, anom, dev)
print(f"clean  : median={np.median(err_clean):.4f}  p95={np.percentile(err_clean,95):.4f}")
print(f"spiked : median={np.median(err_anom):.4f}  p95={np.percentile(err_anom,95):.4f}")
print(f"separation (spiked median / clean p95): {np.median(err_anom)/np.percentile(err_clean,95):.2f}")

/Users/alex/Projects/anomaly-detection-sc/src/models/autoencoder.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(out / "model.pt", map

loaded AE, latent = 4
clean  : median=0.1853  p95=0.6367
spiked : median=0.7238  p95=1.5941
separation (spiked median / clean p95): 1.14


In [7]:
thr = np.percentile(err_clean, 95)           # seuil = 5% FP sur les fenêtres propres

anom_spike = clean.copy(); anom_spike[:, L // 2, 0] += 5.0        # pic ponctuel
anom_shift = clean.copy(); anom_shift[:, L // 2:, 0] += 3.0       # décalage soutenu (2e moitié)

for name, A in [("spike (+5 @center)", anom_spike), ("shift (+3 @2nd half)", anom_shift)]:
    e = reconstruction_error(model, A, dev)
    print(f"{name:22s} median={np.median(e):.3f}  recall@5%FP={(e >= thr).mean():.1%}")

spike (+5 @center)     median=0.724  recall@5%FP=61.3%
shift (+3 @2nd half)   median=0.883  recall@5%FP=78.9%


In [8]:
model8, best8 = train_autoencoder(X, epochs=80, patience=8, latent=8)
print("latent=8 best val:", round(best8, 4))

err_clean8 = reconstruction_error(model8, clean, dev)
thr8 = np.percentile(err_clean8, 95)
for name, A in [("spike", anom_spike), ("shift", anom_shift)]:
    e = reconstruction_error(model8, A, dev)
    print(f"L8 {name:7s} recall@5%FP={(e >= thr8).mean():.1%}")

epoch  1 train 0.1518 val 0.1112
epoch  2 train 0.1045 val 0.0971
epoch  3 train 0.0972 val 0.0943
epoch  4 train 0.0889 val 0.0847
epoch  5 train 0.0846 val 0.0841
epoch  6 train 0.0832 val 0.0811
epoch  7 train 0.0803 val 0.0759
epoch  8 train 0.0730 val 0.0701
epoch  9 train 0.0706 val 0.0689
epoch 10 train 0.0673 val 0.0621
epoch 11 train 0.0609 val 0.0587
epoch 12 train 0.0585 val 0.0572
epoch 13 train 0.0571 val 0.0561
epoch 14 train 0.0560 val 0.0550
epoch 15 train 0.0551 val 0.0537
epoch 16 train 0.0541 val 0.0529
epoch 17 train 0.0536 val 0.0530
epoch 18 train 0.0522 val 0.0517
epoch 19 train 0.0516 val 0.0515
epoch 20 train 0.0510 val 0.0497
epoch 21 train 0.0504 val 0.0492
epoch 22 train 0.0495 val 0.0500
epoch 23 train 0.0492 val 0.0490
epoch 24 train 0.0484 val 0.0474
epoch 25 train 0.0478 val 0.0473
epoch 26 train 0.0483 val 0.0467
epoch 27 train 0.0470 val 0.0465
epoch 28 train 0.0471 val 0.0461
epoch 29 train 0.0464 val 0.0466
epoch 30 train 0.0458 val 0.0454
epoch 31 t

In [1]:
import pandas as pd
from src.config import DATA_DIR
from src.data.aggregate import get_spark
from src.evaluation.injection import build_injection_pool
from src.models.autoencoder import load_ae, get_device
from src.evaluation.ae_eval import build_ae_eval, recall_by_type_ae

spark = get_spark()
features = spark.read.parquet(str(DATA_DIR / "features.parquet"))
pool_pd = build_injection_pool(features, n_series=3000).select("id", "wm_yr_wk").toPandas()
print("pool:", len(pool_pd), "series")

feats = pd.read_parquet(DATA_DIR / "features.parquet")   # pandas, pour le fenêtrage
dev = get_device()
model, cfg = load_ae()

df = build_ae_eval(feats, pool_pd, model, dev)
print("eval windows:", len(df), "| positives:", int((df.label == 1).sum()))
recall_by_type_ae(df)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 18:08:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


pool: 3000 series


/Users/alex/Projects/anomaly-detection-sc/src/models/autoencoder.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(out / "model.pt", map

eval windows: 693020 | positives: 130547
=== AE recall @ 5% / 1% FP ===
spike        k=2.0: recall@5%=10.0%  recall@1%=2.1%
spike        k=2.5: recall@5%=12.3%  recall@1%=3.1%
spike        k=3.0: recall@5%=14.4%  recall@1%=3.9%
spike        k=3.5: recall@5%=16.5%  recall@1%=4.7%
spike        k=4.0: recall@5%=18.4%  recall@1%=5.2%
drop         k=2.0: recall@5%=5.2%  recall@1%=0.8%
drop         k=2.5: recall@5%=7.0%  recall@1%=0.8%
drop         k=3.0: recall@5%=6.7%  recall@1%=1.2%
drop         k=3.5: recall@5%=6.7%  recall@1%=1.4%
drop         k=4.0: recall@5%=8.4%  recall@1%=2.2%
contextual      : recall@5%=5.6%  recall@1%=1.0%
level_shift  k=2.0: recall@5%=18.7%  recall@1%=4.8%
level_shift  k=2.5: recall@5%=28.3%  recall@1%=8.4%
level_shift  k=3.0: recall@5%=38.3%  recall@1%=14.2%
level_shift  k=3.5: recall@5%=49.1%  recall@1%=21.4%
level_shift  k=4.0: recall@5%=59.5%  recall@1%=29.7%


In [3]:
from src.models.autoencoder import log_ae_run, load_ae

model, cfg = load_ae()
run_id = log_ae_run(model, cfg, cfg["best_val"], df)
print("logged AE run:", run_id)

/Users/alex/Projects/anomaly-detection-sc/src/models/autoencoder.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(out / "model.pt", map

logged AE run: d5b0ffc11775486a9259f689b811ab3b


In [1]:
import pandas as pd
from src.config import DATA_DIR
from src.data.aggregate import get_spark
from src.evaluation.injection import build_injection_pool
from src.models.isolation_forest import load_bundle, score_rows, FEATURE_COLS
from src.models.autoencoder import load_ae, get_device
from src.evaluation.consensus_eval import build_consensus_eval, recall_table

# même pool d'injection que l'IF et l'AE
spark = get_spark()
features = spark.read.parquet(str(DATA_DIR / "features.parquet"))
pool_pd = build_injection_pool(features, n_series=3000).select("id", "wm_yr_wk").toPandas()

# IF bundle + scores (pour les négatifs)
feats = pd.read_parquet(DATA_DIR / "features.parquet")
if_model, bounds, thr, if_cols = load_bundle("isolation_forest")
if_thr = float(thr["threshold"])
valid = feats.dropna(subset=FEATURE_COLS).copy()
valid["anomaly_score"] = score_rows(valid, if_model, bounds, if_thr, FEATURE_COLS)["anomaly_score"].values

dev = get_device()
ae_model, _ = load_ae()

df = build_consensus_eval(feats, valid, pool_pd, (if_model, bounds, if_cols),
                          if_thr, ae_model, dev)
print("rows:", len(df), "| positives:", int((df.label == 1).sum()))
recall_table(df)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 18:26:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/Users/alex/Projects/anomaly-detection-sc/src/models/autoencoder.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe

rows: 490627 | positives: 122624
=== recall @ 5% FP  —  IF / AE / consensus ===
spike k=2.0       IF=26.7%  AE=11.0%  consensus=22.1%
spike k=2.5       IF=37.9%  AE=13.6%  consensus=30.4%
spike k=3.0       IF=54.6%  AE=15.6%  consensus=42.3%
spike k=3.5       IF=67.0%  AE=16.9%  consensus=54.5%
spike k=4.0       IF=80.6%  AE=19.1%  consensus=67.6%
drop k=2.0        IF= 5.3%  AE= 6.2%  consensus= 4.2%
drop k=2.5        IF=10.3%  AE= 8.3%  consensus= 6.2%
drop k=3.0        IF=13.2%  AE= 7.7%  consensus= 9.1%
drop k=3.5        IF=15.7%  AE= 8.1%  consensus= 9.6%
drop k=4.0        IF=19.0%  AE=10.2%  consensus=11.9%
contextual        IF=12.1%  AE= 6.6%  consensus=10.6%
level_shift k=2.0  IF=13.3%  AE=20.5%  consensus=19.0%
level_shift k=2.5  IF=16.3%  AE=30.3%  consensus=27.4%
level_shift k=3.0  IF=19.3%  AE=40.5%  consensus=37.2%
level_shift k=3.5  IF=22.6%  AE=51.2%  consensus=47.0%
level_shift k=4.0  IF=25.9%  AE=61.5%  consensus=57.2%


In [2]:
from src.evaluation.consensus_eval import tier_distribution, log_consensus_run

# Répartition des anomalies injectées par tier — valide que les tiers "hintent" le type
tier_distribution(df)

# Log du consensus dans MLflow (même expérience que IF et AE)
run_id = log_consensus_run(df)
print("logged consensus run:", run_id)

tier           both  if_only  ae_only   none
anomaly_type                                
contextual       21      298      153   2156
drop             26      321      236   2943
level_shift   10339     9791    31835  51365
spike          1412     5600      590   5538


/Users/alex/Projects/anomaly-detection-sc/.venv/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


logged consensus run: df0b1509c29045468d1f1c8dedba8581
